# 02c · scFoundation embeddings

**Environment:** `envs/scfoundation.yml` (this foundation model has heavy, specific dependencies: run this notebook in its own environment).
**Output:** one `.h5ad` per dataset whose `X` is the frozen embedding (768-d) and whose `obs` carries `cell_id`, `cell_type`, `master_split`.
The backbone is frozen: no fine-tuning, no gradients.

Get the model: clone `https://github.com/biomap-research/scFoundation`; you need `model/models/models.ckpt` and `OS_scRNA_gene_index.19264.tsv`.
**Important:** scFoundation identifies genes by their position in its fixed 19,264-gene list. The extractor re-orders your genes to that list for you (it prints the coverage); do not skip that step.

> These extraction functions are ported from the authors' working notebooks and need the model's own code and checkpoint.
> They are not run in the repository's tests. Try them on the smallest dataset first and check the shapes at the end.

In [ ]:
from pathlib import Path
from heterosc.extraction.scfoundation import extract_scfoundation_cells

DATA = Path("data")                       # from notebook 01
OUT = Path("outputs/embeddings")          # shared layout: {dataset}/{expert}/{dataset}_{expert}_with_embeddings.h5ad
DATASETS = ["Myeloid"]                    # start small; then add Lung, Liver, Pancreas, Skin
SCF_REPO = Path("checkpoints/scFoundation")
GENE_INDEX = SCF_REPO / "OS_scRNA_gene_index.19264.tsv"

## Part A · annotation datasets

In [ ]:
for ds in DATASETS:
    extract_scfoundation_cells(
        DATA / "annotation" / "processed" / f"{ds}_ready_master_split.h5ad",
        OUT / ds / "scfoundation" / f"{ds}_scfoundation_with_embeddings.h5ad",
        scf_repo=SCF_REPO, gene_index_tsv=GENE_INDEX)

## Part B · cell lines for drug response
Uses `data/drug_response/cellline_expression.h5ad` (CCLE expression, 697 genes). Output layout used by notebook 04:
`outputs/drug_response_embeddings/{expert}/cellline_{expert}_embeddings.h5ad`.

In [ ]:
extract_scfoundation_cells(DATA / "drug_response" / "cellline_expression.h5ad",
                           "outputs/drug_response_embeddings/scfoundation/cellline_scfoundation_embeddings.h5ad",
                           scf_repo=SCF_REPO, gene_index_tsv=GENE_INDEX, extra_obs=())

## Check

In [ ]:
from heterosc.extraction import load_embeddings
import glob
for f in sorted(glob.glob("outputs/**/*_scfoundation_*embeddings.h5ad", recursive=True)):
    X, obs = load_embeddings(f)
    print(f, X.shape, "(expected dim 768)")